# NB01 — Verify Datasets on Kaggle

**Runs on Kaggle CPU (no GPU needed).**

**Datasets to attach:**
- `dummyirl/6isprs` — Potsdam GeoTIFF + labels
- `harish77718/darmstadt-dop20-presliced` — Hessen DOP20 pre-sliced PNG patches

**What this checks:**
1. Potsdam: list tiles under `6ISPRS/`, confirm `6_15` present, inspect label format (indexed vs RGB)
2. DOP20: confirm path, count patches, check channel count, preview one patch
3. Print class lists for both configs

**Kaggle path rule (for datasets added via slug):** `/kaggle/input/datasets/{username}/{dataset-name}/`  
`dummyirl/6isprs` → `/kaggle/input/datasets/dummyirl/6isprs/`

Run this before NB02/NB03 to confirm datasets are attached correctly.

In [9]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


## 1 — Potsdam dataset

In [12]:
from pathlib import Path

POTSDAM = Path("/kaggle/input/datasets/dummyirl/6isprs")
VAL_TILE = "6_15"

tifs = sorted(POTSDAM.rglob("*.tif"))
print(f"Total .tif files: {len(tifs)}")
print()
for p in tifs:
    print(" ", p.relative_to(POTSDAM))

Total .tif files: 12

  top_potsdam_5_14_RGB.tif
  top_potsdam_5_14_label_noBoundary.tif
  top_potsdam_5_15_RGB.tif
  top_potsdam_5_15_label_noBoundary.tif
  top_potsdam_6_13_RGB.tif
  top_potsdam_6_13_label_noBoundary.tif
  top_potsdam_6_14_RGB.tif
  top_potsdam_6_14_label_noBoundary.tif
  top_potsdam_6_15_RGB.tif
  top_potsdam_6_15_label_noBoundary.tif
  top_potsdam_7_13_RGB.tif
  top_potsdam_7_13_label_noBoundary.tif


In [13]:
img_6_15   = sorted(POTSDAM.rglob(f"*{VAL_TILE}*_RGB.tif"))
label_6_15 = sorted(POTSDAM.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"))

print("Val tile image: ", img_6_15[0].name   if img_6_15   else "NOT FOUND")
print("Val tile label: ", label_6_15[0].name if label_6_15 else "NOT FOUND")

Val tile image:  top_potsdam_6_15_RGB.tif
Val tile label:  top_potsdam_6_15_label_noBoundary.tif


In [14]:
import numpy as np
from PIL import Image

# CRITICAL: indexed (single-channel 0-5) or RGB-colored?
# This determines whether NB02 needs a color→index conversion step.
if label_6_15:
    lbl = Image.open(label_6_15[0])
    arr = np.array(lbl)
    print(f"Label file:  {label_6_15[0].name}")
    print(f"PIL mode:    {lbl.mode}")
    print(f"Shape:       {arr.shape}")
    print(f"Dtype:       {arr.dtype}")
    print(f"Unique vals: {np.unique(arr)[:20]}  (showing up to 20)")
    print()
    if arr.ndim == 2 or (arr.ndim == 3 and arr.shape[2] == 1):
        u = np.unique(arr.ravel())
        print(f">>> INDEXED label — {len(u)} unique class indices: {u}")
        print("    NB02 staging: no conversion needed.")
    elif arr.ndim == 3 and arr.shape[2] == 3:
        print(">>> RGB label — needs color→index conversion before eval.")
        print("    Standard ISPRS Potsdam RGB→index map:")
        for rgb, idx, name in [
            ((255,255,255), 0, "impervious surface"),
            ((  0,  0,255), 1, "building"),
            ((  0,255,255), 2, "low vegetation"),
            ((  0,255,  0), 3, "tree"),
            ((255,255,  0), 4, "car"),
            ((255,  0,  0), 5, "clutter"),
        ]:
            print(f"      {str(rgb):20s} → {idx}  {name}")
        print("\n    NB02 will need a conversion cell before eval.")
    else:
        print(f"Unexpected label shape — check manually.")

Label file:  top_potsdam_6_15_label_noBoundary.tif
PIL mode:    RGB
Shape:       (6000, 6000, 3)
Dtype:       uint8
Unique vals: [  0 255]  (showing up to 20)

>>> RGB label — needs color→index conversion before eval.
    Standard ISPRS Potsdam RGB→index map:
      (255, 255, 255)      → 0  impervious surface
      (0, 0, 255)          → 1  building
      (0, 255, 255)        → 2  low vegetation
      (0, 255, 0)          → 3  tree
      (255, 255, 0)        → 4  car
      (255, 0, 0)          → 5  clutter

    NB02 will need a conversion cell before eval.


In [15]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if img_6_15 and label_6_15:
    img_arr = np.array(Image.open(img_6_15[0]))
    lbl_arr = np.array(Image.open(label_6_15[0]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_arr[:, :, :3] if img_arr.ndim == 3 else img_arr)
    axes[0].set_title(f"RGB — {img_6_15[0].name}")
    axes[0].axis("off")
    axes[1].imshow(lbl_arr if lbl_arr.ndim == 2 else lbl_arr[:, :, :3])
    axes[1].set_title(f"Label — {label_6_15[0].name}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/nb01_potsdam_preview.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: /kaggle/working/nb01_potsdam_preview.png")

Saved: /kaggle/working/nb01_potsdam_preview.png


## 2 — Hessen DOP20 dataset

Expected structure: `Darmstadt_dop20_presliced/darmstadt_dop20/images/*.png`  
Patches are pre-sliced 256×256 PNGs (~1267 files). Likely RGB (3-ch), not RGBI.

In [16]:
DOP20_IMAGES = Path("/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images")

patches = sorted(DOP20_IMAGES.glob("*.png"))
print(f"Found {len(patches)} PNG patches")
for p in patches[:5]:
    print(" ", p.name)
if len(patches) > 5:
    print(f"  ... and {len(patches) - 5} more")

Found 1296 PNG patches
  dop20_32_474_5532_1_he_y0_x0.png
  dop20_32_474_5532_1_he_y0_x1024.png
  dop20_32_474_5532_1_he_y0_x1280.png
  dop20_32_474_5532_1_he_y0_x1536.png
  dop20_32_474_5532_1_he_y0_x1792.png
  ... and 1291 more


In [17]:
if patches:
    sample = Image.open(patches[0])
    arr = np.array(sample)
    print(f"File:    {patches[0].name}")
    print(f"Mode:    {sample.mode}")
    print(f"Shape:   {arr.shape}  (H x W x C)")
    print(f"Dtype:   {arr.dtype}")
    print(f"Min/Max: {arr.min()} / {arr.max()}")
    print()
    if arr.ndim == 3 and arr.shape[2] == 4:
        print(">>> RGBI (4 channels). NB03 must drop channel 3 (NIR) before SAM3.")
    elif arr.ndim == 3 and arr.shape[2] == 3:
        print(">>> RGB (3 channels) — NIR already dropped in pre-slicing. NB03 can use as-is.")
    else:
        print(f"Unexpected shape — investigate.")

File:    dop20_32_474_5532_1_he_y0_x0.png
Mode:    RGB
Shape:   (512, 512, 3)  (H x W x C)
Dtype:   uint8
Min/Max: 25 / 255

>>> RGB (3 channels) — NIR already dropped in pre-slicing. NB03 can use as-is.


In [18]:
if patches:
    arr = np.array(Image.open(patches[0]))
    rgb = arr[:, :, :3] if arr.ndim == 3 else arr

    HESSEN_CLASSES = ["agricultural field", "forest", "building",
                      "road", "water body", "background"]
    HESSEN_COLOR_MAP = np.array([
        [210, 180, 140], [34, 139, 34], [220, 20, 60],
        [105, 105, 105], [30, 144, 255], [200, 200, 200],
    ], dtype=np.uint8)

    from matplotlib.patches import Patch
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(rgb)
    axes[0].set_title(f"Preview — {patches[0].name}")
    axes[0].axis("off")

    legend_img = np.zeros((len(HESSEN_CLASSES) * 40, 200, 3), dtype=np.uint8)
    for i, col in enumerate(HESSEN_COLOR_MAP):
        legend_img[i*40:(i+1)*40] = col
    axes[1].imshow(legend_img)
    axes[1].set_yticks([i*40+20 for i in range(len(HESSEN_CLASSES))])
    axes[1].set_yticklabels(HESSEN_CLASSES)
    axes[1].set_xticks([])
    axes[1].set_title("Hessen class colors")

    plt.tight_layout()
    plt.savefig("/kaggle/working/nb01_dop20_preview.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: /kaggle/working/nb01_dop20_preview.png")

Saved: /kaggle/working/nb01_dop20_preview.png


## 3 — Class lists

In [19]:
HESSEN_CLASSES  = ["agricultural field", "forest", "building", "road", "water body", "background"]
POTSDAM_CLASSES = ["impervious surface", "building", "low vegetation", "tree", "car", "clutter"]

print("=== Hessen 6-class (cls_hessen.txt — enriched multi-synonym in fork) ===")
for i, c in enumerate(HESSEN_CLASSES):  print(f"  [{i}] {c}")

print()
print("=== Potsdam 6-class (cls_potsdam.txt — enriched multi-synonym in fork) ===")
for i, c in enumerate(POTSDAM_CLASSES): print(f"  [{i}] {c}")

print()
print("Full synonym prompts: configs/cls_hessen.txt and configs/cls_potsdam.txt")
print("in HarishDeepak/SegEarth-OV-3 (cloned in NB02/NB03).")

=== Hessen 6-class (cls_hessen.txt — enriched multi-synonym in fork) ===
  [0] agricultural field
  [1] forest
  [2] building
  [3] road
  [4] water body
  [5] background

=== Potsdam 6-class (cls_potsdam.txt — enriched multi-synonym in fork) ===
  [0] impervious surface
  [1] building
  [2] low vegetation
  [3] tree
  [4] car
  [5] clutter

Full synonym prompts: configs/cls_hessen.txt and configs/cls_potsdam.txt
in HarishDeepak/SegEarth-OV-3 (cloned in NB02/NB03).


## 4 — Summary checklist

After running all cells above, confirm before proceeding:

**For NB02 (Potsdam eval):**
- [ ] `6_15` image and label found under `6ISPRS/`
- [ ] Label is **indexed** → NB02 staging works as-is
- [ ] Label is **RGB** → NB02 needs a color→index conversion cell (map printed above)

**For NB03 (Hessen inference):**
- [ ] ~1267 PNG patches found under `Darmstadt_dop20_presliced/darmstadt_dop20/images/`
- [ ] Patches are **RGB (3-ch)** → pass directly to SAM3, no channel drop needed
- [ ] Patches are **RGBI (4-ch)** → NB03 must drop channel 3 before SAM3
- [ ] Patches are 256×256 → run each patch directly, no sliding window needed